# PDP-CR trên instance của Thảo

Chọn runtime **CPU** trong Colab và bấm **Runtime → Run all**. Ô 1 nạp mã và dữ liệu từ GitHub. Ô 2 chạy **5 seed song song cho từng thuật toán**, mỗi seed **600 giây CPU**. Có thể đổi `CPU_SECONDS`, `SEEDS` và `WORKERS` trong ô 2.

TSR Thảo, TSR Đức, TSR Claude và GA v2 đều nhận cùng nghiệm greedy ban đầu trên `lr108-5-uniform.csv` gồm 50 tác vụ và 10 robot. Ba TSR dùng chung các giá trị tham số. GA dùng nghiệm chung làm cá thể đầu tiên.

`summary.csv` có makespan trung bình, tốt nhất và seed tốt nhất của mỗi thuật toán. `runs.csv` ghi kết quả từng lượt. Các file JSON chứa nghiệm từng seed và thư mục `logs` chứa nhật ký theo lượt chạy.

In [ ]:
# 1. Nạp mã và chuẩn bị môi trường
from collections import deque
from hashlib import sha256
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPO = "https://github.com/dhducpro/colab_test.git"
CODE_COMMIT = "562c74f73c408091d190ec69a13aa757123b5eba"
workspace = Path(tempfile.mkdtemp(prefix="pdpcr_colab_", dir=Path.cwd()))
code_dir = workspace / "colab_test"

def run_checked(command, env=None):
    recent = deque(maxlen=35)
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace") as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            recent.append(line)
        status = process.wait()
    if status:
        raise RuntimeError(f"{command[0]} dừng với mã {status}.\n{''.join(recent)}")

run_checked(["git", "clone", "--quiet", "--no-checkout", "--config", "core.autocrlf=false", REPO, str(code_dir)])
run_checked(["git", "-C", str(code_dir), "checkout", "--quiet", "--detach", CODE_COMMIT])
dataset = code_dir / "data" / "lr108-5-uniform.csv"
assert sha256(dataset.read_bytes()).hexdigest() == "e3da6196bc21af771d93bfadf8f3ea7ad87c0e9b690ef47c6cc028ce96576343"

python_env = dict(os.environ, PYTHONIOENCODING="utf-8", OPENBLAS_NUM_THREADS="1",
                  OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", NUMEXPR_NUM_THREADS="1")
packages = dict(line.split("==") for line in (code_dir / "requirements.txt").read_text().splitlines() if line)
probe = (
    "import numpy, networkx, pandas; from importlib.metadata import version; "
    f"assert all(version(name) == expected for name, expected in {packages!r}.items())"
)
if subprocess.run([sys.executable, "-c", probe], env=python_env, capture_output=True).returncode:
    dependencies = workspace / "dependencies"
    run_checked([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                 "--target", str(dependencies), "-r", str(code_dir / "requirements.txt")], env=python_env)
    inherited = python_env.get("PYTHONPATH")
    python_env["PYTHONPATH"] = os.pathsep.join([str(dependencies), inherited] if inherited else [str(dependencies)])
run_checked([sys.executable, "-c", probe], env=python_env)
print("Đã nạp mã, dữ liệu và kiểm tra các gói Python.")

In [ ]:
# 2. Chạy thí nghiệm
CPU_SECONDS = 600
SEEDS = [0, 1, 2, 3, 4]
WORKERS = 5

configuration = json.dumps([CODE_COMMIT, float(CPU_SECONDS), SEEDS, WORKERS])
run_id = sha256(configuration.encode("utf-8")).hexdigest()[:12]
out = Path.cwd() / "pdpcr_results" / run_id
print(f"Kết quả được ghi theo từng lượt tại {out}")
try:
    run_checked(
        [sys.executable, "-u", str(code_dir / "run_benchmark.py"), "--seconds", str(CPU_SECONDS),
         "--workers", str(WORKERS), "--output", str(out), "--seeds", *map(str, SEEDS)],
        env=python_env,
    )
except RuntimeError as error:
    raise RuntimeError(f"{error}\nLượt đã hoàn tất được lưu ở {out}. Chạy lại ô 2 để tiếp tục.") from None

In [ ]:
# 3. Xem bảng và tải kết quả
import csv
from shutil import make_archive
from IPython.display import Markdown, display

def show_csv(path, columns):
    if not path.is_file():
        print(f"Chưa có {path.name}.")
        return
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join("---" for _ in columns) + " |"]
    with path.open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            for name in ("initial_cmax", "makespan", "best_makespan", "mean_makespan", "sd_makespan", "cpu_s"):
                if row.get(name):
                    row[name] = f"{float(row[name]):.4f}"
            lines.append("| " + " | ".join(row[name] for name in columns) + " |")
    display(Markdown("\n".join(lines)))

show_csv(out / "summary.csv", ["method", "seeds", "mean_makespan", "best_makespan", "best_seed", "sd_makespan"])
show_csv(out / "runs.csv", ["method", "seed", "initial_cmax", "makespan", "cpu_s"])

if out.is_dir():
    results_zip = make_archive(str(out), "zip", root_dir=out)
    try:
        from google.colab import files
    except ImportError:
        print(results_zip)
    else:
        files.download(results_zip)